In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
import re
import html
import random
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Reproducibility ----
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()

# ---- Paths ----
PROJECT_DIR = Path("/content/drive/MyDrive/SemEval-2025-Task-9")
RAW_PATH    = PROJECT_DIR / "food_recall_incidents.csv"
OUT_DIR     = PROJECT_DIR / "outputs"
CLEAN_DIR   = OUT_DIR / "01_clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# ---- Data ----
LABEL_COLS = ["hazard-category", "product-category", "hazard", "product"]
SPLIT_COL  = "semeval-split"
# Map our split names to the values used in the `semeval-split` column
SPLIT_VALUES = {"train": "training", "valid": "validation", "test": "test"}
EXPECTED_SIZES = {"train": 5082, "valid": 565, "test": 997}

# Set to True to ignore cached files on Drive and redo a stage
FORCE_RECOMPUTE = False

# ---- Language filter ----
KEEP_LANGUAGES = {"en"}                   # check the exact code with value_counts below
LANGUAGE_FILTER_SPLITS = ["train"]        # add "valid", "test" to filter those too

# ---- Augmentation ----
AUG_DIR          = OUT_DIR / "02_augment"
AUG_DIR.mkdir(parents=True, exist_ok=True)
AUG_DRIVER_LABEL = "product"   # exact product label drives augmentation
AUG_TARGET       = 10          # fill each rare class up to this many samples
AUG_ALPHA        = 0.1         # fraction of words changed per variant (EDA default)

# ---- Encoder training ----
MODEL_NAME     = "roberta-base"            # or "microsoft/deberta-v3-base"
ENC_DIR        = OUT_DIR / "03_encoder"
LOCAL_CKPT_DIR = Path("/content/checkpoints")   # runtime disk, faster than Drive
MAX_LEN        = 512
BATCH_SIZE     = 16
LR             = 1e-5                      # encoder learning rate (SRCB)
HEAD_LR        = 1e-4                      # classifier head; set = LR to follow SRCB exactly
MAX_EPOCHS     = 10
PATIENCE       = 2                         # early stopping on validation macro-F1
FOCAL_ALPHA    = 0.25
FOCAL_GAMMA    = 2.0
CAND_THRESHOLD = 0.5
CAND_TOP_K     = 5                         # fallback when nothing passes the threshold
CAND_MAX       = 15                        # cap on candidate list length

# ---- LLM prediction ----
LLM_DIR          = OUT_DIR / "04_llm"
LLM_DIR.mkdir(parents=True, exist_ok=True)
LIST_ALL_ST1     = True    # ST1: list all categories (ordered by BERT probability)
MAX_PROMPT_WORDS = 600     # cap very long texts to keep cost predictable
LLM_WORKERS      = 8       # parallel API requests
LLM_MAX_RETRIES  = 5
PILOT_PER_TASK   = 25      # 4 tasks x 25 = 100 pilot prompts

In [25]:
# Spaces, tabs, non-breaking and zero-width spaces (but not newlines)
_INLINE_WS = re.compile(r"[ \t\u00a0\u200b\u2009\u202f]+")
_NON_WORD  = re.compile(r"\W+")

def _to_str(x) -> str:
    return "" if pd.isna(x) else str(x)

def normalise_whitespace(s: str) -> str:
    """Unescape HTML entities, unify line breaks, collapse spaces, drop empty lines."""
    s = html.unescape(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = _INLINE_WS.sub(" ", s)
    lines = (line.strip() for line in s.split("\n"))
    return "\n".join(line for line in lines if line)

def _segment_key(segment: str) -> str:
    """Comparison key: ignore punctuation and spacing differences."""
    return _NON_WORD.sub(" ", segment).strip()

def clean_title_text(title, text):
    """Clean title and text separately; remove text lines that repeat
    the title or an earlier line. Returns (clean_title, clean_text)."""
    title = normalise_whitespace(_to_str(title)).lower().replace("\n", " ")
    text  = normalise_whitespace(_to_str(text)).lower()

    seen = {_segment_key(title)} if title else set()
    kept = []
    for segment in text.split("\n"):
        key = _segment_key(segment)
        if key and key not in seen:
            seen.add(key)
            kept.append(segment)
    return title, "\n".join(kept)

def build_input(title: str, text: str) -> str:
    """Model input: title on the first line, text below."""
    return f"{title}\n{text}".strip()

In [26]:
clean_paths = {name: CLEAN_DIR / f"{name}.parquet" for name in SPLIT_VALUES}

if not FORCE_RECOMPUTE and all(p.exists() for p in clean_paths.values()):
    splits = {name: pd.read_parquet(p) for name, p in clean_paths.items()}
    print("Loaded cleaned splits from Drive.")
else:
    df = pd.read_csv(RAW_PATH)
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    df["sample_id"] = df.index                      # link back to the raw file

    splits = {}
    for name, value in SPLIT_VALUES.items():
        part = df[df[SPLIT_COL] == value].copy()

        # 1) Language filter
        if name in LANGUAGE_FILTER_SPLITS:
            mask = part["language"].isin(KEEP_LANGUAGES)
            removed = part.loc[~mask, "language"].value_counts().to_dict()
            part = part[mask]
            print(f"{name}: removed {(~mask).sum()} non-English rows {removed}")

        # 2) Check against the official split size
        if len(part) != EXPECTED_SIZES[name]:
            raise ValueError(
                f"{name}: expected {EXPECTED_SIZES[name]} rows, got {len(part)}. "
                f"Values in '{SPLIT_COL}': {sorted(df[SPLIT_COL].unique())}"
            )

        # 3) Clean
        cleaned = [clean_title_text(t, x) for t, x in zip(part["title"], part["text"])]
        part["clean_title"] = [t for t, _ in cleaned]
        part["clean_text"]  = [x for _, x in cleaned]
        part["input_text"]  = [build_input(t, x) for t, x in cleaned]

        splits[name] = part.reset_index(drop=True)

    for name, part in splits.items():          # save only after all splits pass
        part.to_parquet(clean_paths[name], index=False)
    print("Cleaned and saved splits to Drive.")

train_df, val_df, test_df = splits["train"], splits["valid"], splits["test"]

train: removed 902 non-English rows {'de': 888, 'fr': 8, 'gr': 4, 'dk': 1, 'it': 1}
Cleaned and saved splits to Drive.


In [27]:
for name, part in splits.items():
    assert len(part) == EXPECTED_SIZES[name], f"{name}: {len(part)} rows"
    raw_len   = (part["title"].fillna("").str.len() + part["text"].fillna("").str.len())
    clean_len = part["input_text"].str.len()
    reduction = 1 - clean_len / raw_len.clip(lower=1)
    words     = part["input_text"].str.split().str.len()
    print(f"{name:5s} | rows {len(part):5d} | empty text {(part['clean_text'] == '').sum():3d} | "
          f"median chars removed {reduction.median():.1%} | rows with >20% removed {(reduction > 0.2).sum():4d} | "
          f"words p50/p90/max {int(words.median())}/{int(words.quantile(0.9))}/{words.max()}")

# Eyeball the training sample where cleaning removed the most
raw_len   = train_df["title"].str.len() + train_df["text"].str.len()
i = (1 - train_df["input_text"].str.len() / raw_len).idxmax()
print("\n--- RAW ---\n", train_df.loc[i, "title"], "\n", train_df.loc[i, "text"][:1500])
print("\n--- CLEAN ---\n", train_df.loc[i, "input_text"][:1500])

train | rows  5082 | empty text   2 | median chars removed 0.0% | rows with >20% removed  758 | words p50/p90/max 299/516/8416
valid | rows   565 | empty text   0 | median chars removed 0.0% | rows with >20% removed   87 | words p50/p90/max 292/535/1615
test  | rows   997 | empty text   0 | median chars removed 0.1% | rows with >20% removed  152 | words p50/p90/max 293/531/7267

--- RAW ---
 Polycyclic aromatic hydrocarbons in olive pomace oil by Grifo, San Remo, La Molisana from Italy 
 Newsroom > Food Recalls and Allergy Alerts > Archives 
This archive of previously issued food recalls and allergy alerts is provided for reference and research purposes.
Users should note that the products listed in the archive have been subject to removal from the marketplace or appropriate corrective action. Food recalls or allergy alerts are not an indication of the food safety status of products produced at a later date.
OTTAWA, October 9, 2001 - The Canadian Food Inspection Agency (CFIA) is warnin

In [33]:
import json
import difflib
import datetime
from functools import lru_cache

import nltk
for pkg in ["wordnet", "omw-1.4", "stopwords",
            "averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"]:
    nltk.download(pkg, quiet=True)
from nltk.corpus import wordnet, stopwords

STOPWORDS = set(stopwords.words("english"))
_PENN_TO_WN = {"N": wordnet.NOUN, "V": wordnet.VERB, "J": wordnet.ADJ, "R": wordnet.ADV}

@lru_cache(maxsize=None)
def get_synonyms(word, pos):
    """Synonyms from the word's most frequent WordNet sense for its part of speech.
    Avoids rare senses like egg -> 'testicle'. Sorted for reproducibility."""
    if pos is None:
        return []
    synsets = wordnet.synsets(word, pos=pos)
    if not synsets:
        return []
    syns = set()
    for lemma in synsets[0].lemmas():
        s = lemma.name().replace("_", " ").replace("-", " ").lower()
        s = "".join(c for c in s if c in " abcdefghijklmnopqrstuvwxyz").strip()
        if s and s != word:
            syns.add(s)
    return sorted(syns)

_TOKEN_RE  = re.compile(r"\n|[^\s]+")                 # words (with punctuation) and line breaks
_EDGE_PUNC = re.compile(r"^(\W*)(.*?)(\W*)$", re.S)   # "(sausage)," -> "(", "sausage", "),"
_ALPHA_RE  = re.compile(r"^[a-z]+(?:[-'][a-z]+)*$")
_SENT_END  = re.compile(r"[.!?;:]$")

def _split_edges(token):
    return _EDGE_PUNC.match(token).groups()

def protected_words(*labels):
    """All words of the given labels, e.g. {'listeria', 'monocytogenes', 'smoked', 'sausage'}."""
    words = set()
    for label in labels:
        words.update(w for w in re.split(r"\W+", str(label).lower()) if w)
    return words

def _is_locked(core, protected):
    """Locked if the token contains a protected content word ('walnut' also locks 'walnuts'),
    or exactly matches a protected stopword ('and' in 'milk and products thereof')."""
    words = [w for w in re.split(r"\W+", core) if w]
    for p in protected:
        if p in STOPWORDS:
            if p in words:
                return True
        elif p in core:
            return True
    return False

class Token:
    __slots__ = ("text", "core", "is_break", "locked", "pos", "sent")
    def __init__(self, text, protected):
        self.text = text
        self.is_break = text == "\n"
        self.core = "" if self.is_break else _split_edges(text)[1]
        self.locked = self.is_break or _is_locked(self.core, protected)   # never edited or moved
        self.pos = None
        self.sent = 0

    @property
    def editable(self):
        """Candidate for synonym replacement and source word for insertion."""
        return (not self.locked and self.pos is not None and self.core not in STOPWORDS
                and _ALPHA_RE.match(self.core) is not None)

def _tokenize(text, protected):
    tokens = [Token(t, protected) for t in _TOKEN_RE.findall(text)]
    words = [t for t in tokens if not t.is_break]
    for t, (_, tag) in zip(words, nltk.pos_tag([t.core or t.text for t in words])):
        t.pos = _PENN_TO_WN.get(tag[:1])
    sent = 0                                              # sentence index, for local swaps
    for t in tokens:
        t.sent = sent
        if t.is_break or _SENT_END.search(t.text):
            sent += 1
    return tokens

def _replace_core(token_text, new_core):
    pre, _, suf = _split_edges(token_text)
    return f"{pre}{new_core}{suf}"

def _detok(tokens):
    return re.sub(r" ?\n ?", "\n", " ".join(t.text for t in tokens))

# ---- The four EDA operations (Wei & Zou, 2019), adapted for long texts ----

def synonym_replacement(tokens, n, rng):
    """Replace n individual words (tokens, not word types) with a synonym."""
    candidates = [i for i, t in enumerate(tokens) if t.editable]
    rng.shuffle(candidates)
    replaced = 0
    for i in candidates:
        synonyms = get_synonyms(tokens[i].core, tokens[i].pos)
        if synonyms:
            syn = rng.choice(synonyms)
            tokens[i].text, tokens[i].core = _replace_core(tokens[i].text, syn), syn
            replaced += 1
            if replaced >= n:
                break
    return tokens

def random_insertion(tokens, n, rng):
    """Insert n synonyms of random words at random positions."""
    sources = [t for t in tokens if t.editable]
    for _ in range(n):
        synonyms = []
        for _attempt in range(10):
            if not sources:
                return tokens
            src = rng.choice(sources)
            synonyms = get_synonyms(src.core, src.pos)
            if synonyms:
                break
        if not synonyms:
            continue
        # never insert between two locked words (would split "smoked sausage")
        positions = [i for i in range(len(tokens) + 1)
                     if not (0 < i < len(tokens) and tokens[i - 1].locked and tokens[i].locked
                             and not tokens[i - 1].is_break and not tokens[i].is_break)]
        tokens.insert(rng.choice(positions), Token(rng.choice(synonyms), set()))
    return tokens

def random_swap(tokens, n, rng):
    """Swap n pairs of unlocked words, each pair within the same sentence."""
    by_sent = {}
    for i, t in enumerate(tokens):
        if not t.locked:
            by_sent.setdefault(t.sent, []).append(i)
    pools = [idx for idx in by_sent.values() if len(idx) >= 2]
    if not pools:
        return tokens
    weights = [len(p) for p in pools]
    for _ in range(n):
        i, j = rng.sample(rng.choices(pools, weights=weights)[0], 2)
        tokens[i], tokens[j] = tokens[j], tokens[i]
    return tokens

def random_deletion(tokens, p, rng):
    """Delete each unlocked word with probability p."""
    kept = [t for t in tokens if t.locked or rng.random() > p]
    return kept if any(not t.is_break for t in kept) else tokens

OPERATIONS = ("synonym_replacement", "random_insertion", "random_swap", "random_deletion")

def eda(text, operation, alpha, rng, protected):
    """Apply one EDA operation, changing about `alpha` of the words.
    Protected words and line breaks are never replaced, deleted or moved."""
    tokens = _tokenize(text, protected)
    n_words = sum(not t.is_break for t in tokens)
    if n_words == 0:
        return text
    n = max(1, int(alpha * n_words))
    if operation == "synonym_replacement":
        tokens = synonym_replacement(tokens, n, rng)
    elif operation == "random_insertion":
        tokens = random_insertion(tokens, n, rng)
    elif operation == "random_swap":
        tokens = random_swap(tokens, n, rng)
    elif operation == "random_deletion":
        tokens = random_deletion(tokens, alpha, rng)
    else:
        raise ValueError(operation)
    return _detok(tokens)

In [34]:
def augment_training_set(train_df, seed=SEED):
    rng = random.Random(seed)
    counts = train_df[AUG_DRIVER_LABEL].value_counts()
    rare = sorted(counts[counts < AUG_TARGET].index)    # sorted -> reproducible order

    new_rows, op_cycle = [], 0
    for label in rare:
        sources = train_df[train_df[AUG_DRIVER_LABEL] == label]
        for k in range(AUG_TARGET - len(sources)):
            src = sources.iloc[k % len(sources)]          # round-robin over the class's samples
            op = OPERATIONS[op_cycle % len(OPERATIONS)]   # rotate through the four operations
            op_cycle += 1
            new_text = eda(src["clean_text"], op, AUG_ALPHA, rng,
                           protected_words(src["hazard"], src["product"]))
            row = src.to_dict()                           # keeps all four labels
            row.update({
                "clean_text": new_text,
                "input_text": build_input(src["clean_title"], new_text),  # title unchanged
                "is_augmented": True,
                "aug_op": op,
                "source_sample_id": src["sample_id"],
            })
            new_rows.append(row)

    originals = train_df.assign(is_augmented=False, aug_op=None,
                                source_sample_id=train_df["sample_id"])
    return pd.concat([originals, pd.DataFrame(new_rows)], ignore_index=True), len(rare)


aug_path    = AUG_DIR / "train_augmented.parquet"
aug_cfg_path = AUG_DIR / "aug_config.json"

if not FORCE_RECOMPUTE and aug_path.exists():
    train_aug_df = pd.read_parquet(aug_path)
    print("Loaded augmented training set from Drive.")
else:
    train_aug_df, n_rare = augment_training_set(train_df)
    train_aug_df.to_parquet(aug_path, index=False)

    aug_rows = train_aug_df[train_aug_df["is_augmented"]]
    record = {                                            # the record the architecture asks for
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "operations": list(OPERATIONS),
        "change_rate_alpha": AUG_ALPHA,
        "target_per_class": AUG_TARGET,
        "driver_label": AUG_DRIVER_LABEL,
        "seed": SEED,
        "protected": "words of each sample's hazard and product labels (incl. inflected forms "
                     "like walnut -> walnuts); title not augmented",
        "adaptations": "POS-tagged, most-frequent-sense WordNet synonyms; alpha counts tokens, "
                       "not word types; swaps restricted to the same sentence",
        "n_rare_classes": n_rare,
        "n_original": int((~train_aug_df["is_augmented"]).sum()),
        "n_augmented": len(aug_rows),
        "per_operation": aug_rows["aug_op"].value_counts().to_dict(),
        "nltk_version": nltk.__version__,
    }
    aug_cfg_path.write_text(json.dumps(record, indent=2))
    print("Augmented and saved to Drive.")
    print(json.dumps(record, indent=2))

Augmented and saved to Drive.
{
  "created": "2026-10-25T22:32:42",
  "operations": [
    "synonym_replacement",
    "random_insertion",
    "random_swap",
    "random_deletion"
  ],
  "change_rate_alpha": 0.1,
  "target_per_class": 10,
  "driver_label": "product",
  "seed": 42,
  "protected": "words of each sample's hazard and product labels (incl. inflected forms like walnut -> walnuts); title not augmented",
  "adaptations": "POS-tagged, most-frequent-sense WordNet synonyms; alpha counts tokens, not word types; swaps restricted to the same sentence",
  "n_rare_classes": 905,
  "n_original": 5082,
  "n_augmented": 6903,
  "per_operation": {
    "synonym_replacement": 1726,
    "random_insertion": 1726,
    "random_swap": 1726,
    "random_deletion": 1725
  },
  "nltk_version": "3.9.1"
}


In [35]:
aug_rows = train_aug_df[train_aug_df["is_augmented"]]
src_text = train_df.set_index("sample_id")["clean_text"]

# 1) Every product class now has at least AUG_TARGET samples
print("Smallest product class after augmentation:",
      train_aug_df[AUG_DRIVER_LABEL].value_counts().min())

# 2) Label phrases survive: each hazard/product phrase occurs at least as often as in the source
violations = [
    (i, lab) for i, r in aug_rows.iterrows()
    for lab in (r["hazard"], r["product"])
    if r["clean_text"].count(lab) < src_text[r["source_sample_id"]].count(lab)
]
print("Label-phrase violations:", len(violations))

# 3) Title unchanged
print("Titles intact:", (aug_rows["input_text"].str.split("\n").str[0]
                         == aug_rows["clean_title"]).all())

# 4) Spot-check: one example per operation, showing only the changed words
def show_changes(original, augmented):
    a, b = original.split(), augmented.split()
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b).get_opcodes():
        if tag != "equal":
            print(f"   {tag:8s} {' '.join(a[i1:i2])!r} -> {' '.join(b[j1:j2])!r}")

for op in OPERATIONS:
    r = aug_rows[aug_rows["aug_op"] == op].sample(1, random_state=SEED).iloc[0]
    print(f"\n=== {op} | product: {r['product']} | hazard: {r['hazard']} ===")
    print(r["clean_text"][:400], "...")
    show_changes(src_text[r["source_sample_id"]], r["clean_text"])

Smallest product class after augmentation: 10
Label-phrase violations: 0
Titles intact: True

=== synonym_replacement | product: flavoured whey protein preparations | hazard: soybeans and products thereof ===
premia health is extending its recall to include protein active shakes (include diet whey and protein), protein active paleo bars and premia health premia whey tubs, because they contain one or more of the following allergen: soya and egg, which is not mentioned on the product labels. this mean the products are a possible health endangerment for anyone with an allergy to soya and/ or egg.
product  ...
   replace  '(including' -> '(include'
   replace  'allergens:' -> 'allergen:'
   replace  'means' -> 'mean'
   replace  'risk' -> 'endangerment'
   replace  '(including' -> '(include'
   insert   '' -> 'drinking'
   replace  'flavours' -> 'tone'
   replace  'soya' -> 'soy'
   replace  'bar pack' -> 'barroom multitude'
   replace  'dates' -> 'date'
   replace  'pack' -> 'multitude'
 

In [37]:
import gc
import time
import torch
from torch.utils.data import DataLoader
from torchvision.ops import sigmoid_focal_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

DEVICE = "cuda"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts):
    return tokenizer(list(texts), truncation=True, max_length=MAX_LEN)["input_ids"]

# Tokenize once, reuse for all four tasks
enc_train = encode(train_aug_df["input_text"])
enc_val   = encode(val_df["input_text"])
enc_test  = encode(test_df["input_text"])

def make_loader(input_ids, label_idx, n_labels, shuffle):
    items = list(zip(input_ids, label_idx))
    def collate(batch):
        ids, labs = zip(*batch)
        padded = tokenizer.pad({"input_ids": list(ids)}, return_tensors="pt")
        target = torch.zeros(len(labs), n_labels)
        for row, lab in enumerate(labs):
            if lab >= 0:                       # -1 = label not seen in training
                target[row, lab] = 1.0
        return padded["input_ids"], padded["attention_mask"], target
    return DataLoader(items, batch_size=BATCH_SIZE if shuffle else 2 * BATCH_SIZE,
                      shuffle=shuffle, collate_fn=collate, num_workers=2)

def build_model(n_labels):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_labels)
    # Focal-loss prior (Lin et al., 2017): start every label at p ~= 0.01 instead of 0.5
    head = model.classifier.out_proj if hasattr(model.classifier, "out_proj") else model.classifier
    torch.nn.init.constant_(head.bias, -float(np.log((1 - 0.01) / 0.01)))
    return model.to(DEVICE)

def focal_loss(logits, target):
    # sum over labels, mean over samples (one positive per sample)
    return sigmoid_focal_loss(logits.float(), target, alpha=FOCAL_ALPHA,
                              gamma=FOCAL_GAMMA, reduction="sum") / logits.size(0)

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    out = []
    for ids, mask, _ in loader:
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

def select_candidates(probs, labels):
    """Labels with p > threshold, highest first; top-k fallback; capped at CAND_MAX."""
    result = []
    for row in probs:
        order = np.argsort(-row)
        above = [i for i in order if row[i] > CAND_THRESHOLD]
        chosen = (above or list(order[:CAND_TOP_K]))[:CAND_MAX]
        result.append([labels[i] for i in chosen])
    return result

def evaluate(probs, labels, gold):
    top1 = [labels[i] for i in probs.argmax(1)]
    cands = select_candidates(probs, labels)
    return {
        "macro_f1_top1": f1_score(gold, top1, average="macro", zero_division=0),
        "candidate_recall": float(np.mean([g in c for g, c in zip(gold, cands)])),
        "avg_candidates": float(np.mean([len(c) for c in cands])),
    }

def train_task(task):
    set_seed()
    labels = sorted(train_df[task].unique())             # vocabulary from training data only
    lab2id = {l: i for i, l in enumerate(labels)}
    to_idx = lambda s: [lab2id.get(l, -1) for l in s]

    train_loader = make_loader(enc_train, to_idx(train_aug_df[task]), len(labels), shuffle=True)
    val_loader   = make_loader(enc_val,   to_idx(val_df[task]),       len(labels), shuffle=False)

    model = build_model(len(labels))
    head_params = [p for n, p in model.named_parameters() if n.startswith("classifier")]
    body_params = [p for n, p in model.named_parameters() if not n.startswith("classifier")]
    optimizer = torch.optim.AdamW([{"params": body_params, "lr": LR},
                                   {"params": head_params, "lr": HEAD_LR}], weight_decay=0.01)
    total_steps = MAX_EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)

    best, best_epoch, history = -1.0, -1, []
    ckpt = LOCAL_CKPT_DIR / task / "best.pt"
    ckpt.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        start, running = time.time(), 0.0
        for step, (ids, mask, target) in enumerate(tqdm(train_loader, desc=f"{task} ep{epoch}", leave=False), 1):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
            loss = focal_loss(logits, target.to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            running += loss.item()

        metrics = evaluate(predict_probs(model, val_loader), labels, list(val_df[task]))
        metrics.update(epoch=epoch, train_loss=running / step, minutes=(time.time() - start) / 60)
        history.append(metrics)
        print(f"{task} | epoch {epoch} | loss {metrics['train_loss']:.4f} | "
              f"val macro-F1 {metrics['macro_f1_top1']:.4f} | cand. recall {metrics['candidate_recall']:.4f} | "
              f"avg cands {metrics['avg_candidates']:.1f} | {metrics['minutes']:.1f} min")

        if metrics["macro_f1_top1"] > best:
            best, best_epoch = metrics["macro_f1_top1"], epoch
            torch.save(model.state_dict(), ckpt)
        elif epoch - best_epoch >= PATIENCE:
            print(f"{task} | early stopping, best epoch {best_epoch}")
            break

    model.load_state_dict(torch.load(ckpt))
    return model, labels, history, best_epoch

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [38]:
for task in LABEL_COLS:
    task_dir = ENC_DIR / task
    if (task_dir / "candidates_test.json").exists() and not FORCE_RECOMPUTE:
        print(f"{task}: already done, skipping.")
        continue
    task_dir.mkdir(parents=True, exist_ok=True)

    model, labels, history, best_epoch = train_task(task)

    # Probabilities and candidate lists for validation and test
    lab2id = {l: i for i, l in enumerate(labels)}
    for split_name, enc, part in [("val", enc_val, val_df), ("test", enc_test, test_df)]:
        loader = make_loader(enc, [lab2id.get(l, -1) for l in part[task]], len(labels), shuffle=False)
        probs = predict_probs(model, loader)
        np.save(task_dir / f"probs_{split_name}.npy", probs.astype(np.float16))
        cands = select_candidates(probs, labels)
        (task_dir / f"candidates_{split_name}.json").write_text(json.dumps(
            [{"sample_id": int(s), "candidates": c} for s, c in zip(part["sample_id"], cands)]))

    (task_dir / "labels.json").write_text(json.dumps(labels))
    (task_dir / "history.json").write_text(json.dumps(
        {"best_epoch": best_epoch, "model": MODEL_NAME, "epochs": history}, indent=2))
    model.save_pretrained(task_dir / "model")             # final weights only, ~0.5 GB
    tokenizer.save_pretrained(task_dir / "model")

    del model
    gc.collect(); torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


hazard-category ep1:   0%|          | 0/750 [00:00<?, ?it/s]

...

product ep9:   0%|          | 0/750 [00:00<?, ?it/s]

product | epoch 9 | loss 0.0404 | val macro-F1 0.3111 | cand. recall 0.6372 | avg cands 2.6 | 3.1 min


product ep10:   0%|          | 0/750 [00:00<?, ?it/s]

product | epoch 10 | loss 0.0361 | val macro-F1 0.3112 | cand. recall 0.6372 | avg cands 2.6 | 3.4 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [40]:
rows = []
for task in LABEL_COLS:
    h = json.loads((ENC_DIR / task / "history.json").read_text())
    best = next(e for e in h["epochs"] if e["epoch"] == h["best_epoch"])
    rows.append({"task": task, "best_epoch": h["best_epoch"],
                 **{k: round(best[k], 4) for k in ["macro_f1_top1", "candidate_recall", "avg_candidates"]}})
pd.DataFrame(rows)

,task,best_epoch,macro_f1_top1,candidate_recall,avg_candidates
0,hazard-category,7,0.8809,0.9558,1.0513
1,product-category,4,0.7169,0.8230,1.2319
2,hazard,3,0.6117,0.8973,1.4248
3,product,10,0.3112,0.6372,2.6478


In [ ]:
# !pip install -q -U google-genai     # uncomment if the import below fails
from google import genai
from google.genai import types

API_KEY   = (PROJECT_DIR / "api_key.txt").read_text().strip()   # key from Google AI Studio
LLM_MODEL = "gemini-3.5-flash-lite"

_client = genai.Client(api_key=API_KEY)
_config = types.GenerateContentConfig(
    temperature=0,
    max_output_tokens=256,
    thinking_config=types.ThinkingConfig(thinking_level="minimal"),   # closest to "off"
)

def call_llm(prompt: str) -> str:
    """One prompt -> raw answer."""
    resp = _client.models.generate_content(model=LLM_MODEL, contents=prompt, config=_config)
    return resp.text or ""

print(repr(call_llm("Answer with one word: what colour is the sky on a clear day?")))

'Blue'


In [44]:
import difflib
import threading
from concurrent.futures import ThreadPoolExecutor

TASK_QUESTION = {
    "hazard-category":  "Which of the following is the category of the food hazard in this report?",
    "product-category": "Which of the following is the category of the food product in this report?",
    "hazard":           "Which of the following is the specific hazard in this report (e.g. a pathogen, allergen or contaminant)?",
    "product":          "Which of the following is the specific food product in this report?",
}
SPLIT_DF = {"val": val_df, "test": test_df}

def cap_words(text, n=MAX_PROMPT_WORDS):
    words = list(re.finditer(r"\S+", text))
    return text if len(words) <= n else text[:words[n - 1].end()] + " ..."

def build_prompt(task, title, text, options):
    numbered = "\n".join(f"{i}. {o}" for i, o in enumerate(options, 1))
    return (
        "You are classifying food recall reports.\n\n"
        f"Title: {title}\n"
        f"Text: {cap_words(text)}\n\n"
        f"{TASK_QUESTION[task]}\n"
        f"{numbered}\n\n"
        "Answer with exactly one label from the list, copied exactly, and nothing else."
    )

def get_options(task, split):
    """Candidate lists aligned with the rows of the split, highest BERT probability first."""
    task_dir = ENC_DIR / task
    if LIST_ALL_ST1 and task in ("hazard-category", "product-category"):
        labels = json.loads((task_dir / "labels.json").read_text())
        probs = np.load(task_dir / f"probs_{split}.npy").astype(np.float32)
        return [[labels[i] for i in np.argsort(-row)] for row in probs]
    cands = json.loads((task_dir / f"candidates_{split}.json").read_text())
    assert [c["sample_id"] for c in cands] == list(SPLIT_DF[split]["sample_id"])
    return [c["candidates"] for c in cands]

def _norm(s):
    return re.sub(r"\s+", " ", s.strip().strip("\"'`*.").strip()).lower()

def parse_answer(response, options):
    """Exact match -> list number -> fuzzy match -> label contained in answer -> top BERT candidate."""
    lines = [l for l in (response or "").splitlines() if l.strip()]
    text = _norm(lines[0]) if lines else ""
    norm_opts = {_norm(o): o for o in options}
    if text in norm_opts:
        return norm_opts[text], "exact"
    m = re.match(r"^(\d+)\s*[.):-]?\s*(.*)$", text)
    if m:
        rest = _norm(m.group(2))
        if rest in norm_opts:
            return norm_opts[rest], "exact"
        if not rest and 1 <= int(m.group(1)) <= len(options):
            return options[int(m.group(1)) - 1], "number"
    close = difflib.get_close_matches(text, list(norm_opts), n=1, cutoff=0.8)
    if close:
        return norm_opts[close[0]], "fuzzy"
    contained = [o for o in norm_opts if o in text]
    if contained:
        return norm_opts[max(contained, key=len)], "contained"
    return options[0], "fallback"

def call_with_retry(prompt):
    for attempt in range(LLM_MAX_RETRIES):
        try:
            return call_llm(prompt)
        except Exception as e:
            wait = 2 ** attempt
            print(f"API error ({type(e).__name__}: {e}); retry in {wait}s")
            time.sleep(wait)
    return None                                  # gives up; retried on the next run

def load_jsonl(path):
    if not path.exists():
        return {}
    with open(path) as f:
        return {r["sample_id"]: r for r in map(json.loads, f)}

def run_llm(task, split, out_dir, limit=None):
    """Run the LLM for one task/split. Appends to a JSONL file and skips samples already done."""
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task}_{split}.jsonl"
    done = load_jsonl(path)
    part = SPLIT_DF[split]
    options = get_options(task, split)

    rows = list(zip(part["sample_id"], part["clean_title"], part["clean_text"], options))[:limit]
    jobs, single = [], []
    for sid, title, text, opts in rows:
        if int(sid) in done:
            continue
        if len(opts) == 1:                       # nothing to choose: no API call needed
            single.append({"sample_id": int(sid), "options": opts, "response": None,
                           "prediction": opts[0], "method": "single_candidate"})
        else:
            jobs.append((int(sid), build_prompt(task, title, text, opts), opts))

    def process(job):
        sid, prompt, opts = job
        response = call_with_retry(prompt)
        if response is None:
            return None
        pred, method = parse_answer(response, opts)
        return {"sample_id": sid, "options": opts, "response": response,
                "prediction": pred, "method": method}

    lock = threading.Lock()
    with open(path, "a") as f:
        for rec in single:
            f.write(json.dumps(rec) + "\n")
        with ThreadPoolExecutor(LLM_WORKERS) as ex:
            for rec in tqdm(ex.map(process, jobs), total=len(jobs), desc=f"{task}/{split}"):
                if rec is not None:
                    with lock:
                        f.write(json.dumps(rec) + "\n"); f.flush()

    results = load_jsonl(path)
    missing = len(rows) - len(results)
    methods = pd.Series([r["method"] for r in results.values()]).value_counts().to_dict()
    print(f"{task}/{split}: {len(results)} done, {missing} missing | {methods}")
    return results

In [45]:
PILOT_DIR = LLM_DIR / "pilot"
for task in LABEL_COLS:
    res = run_llm(task, "val", PILOT_DIR, limit=PILOT_PER_TASK)
    for r in [r for r in res.values() if r["method"] != "single_candidate"][:2]:
        print(f"   raw: {r['response']!r} -> {r['prediction']!r} ({r['method']})")

hazard-category/val:   0%|          | 0/25 [00:00<?, ?it/s]

hazard-category/val: 25 done, 0 missing | {'exact': 25}
   raw: 'biological' -> 'biological' (exact)
   raw: '1. biological' -> 'biological' (exact)


product-category/val:   0%|          | 0/25 [00:00<?, ?it/s]

product-category/val: 25 done, 0 missing | {'exact': 25}
   raw: '1. meat, egg and dairy products' -> 'meat, egg and dairy products' (exact)
   raw: '1. meat, egg and dairy products' -> 'meat, egg and dairy products' (exact)


hazard/val:   0%|          | 0/6 [00:00<?, ?it/s]

hazard/val: 25 done, 0 missing | {'single_candidate': 19, 'exact': 6}
   raw: 'foreign bodies' -> 'foreign bodies' (exact)
   raw: 'yeasts' -> 'yeasts' (exact)


product/val:   0%|          | 0/4 [00:00<?, ?it/s]

product/val: 25 done, 0 missing | {'single_candidate': 21, 'exact': 4}
   raw: 'cooked ham' -> 'cooked ham' (exact)
   raw: '1. cooked chicken' -> 'cooked chicken' (exact)


In [46]:
for split in ["val", "test"]:
    for task in LABEL_COLS:
        run_llm(task, split, LLM_DIR)

hazard-category/val:   0%|          | 0/565 [00:00<?, ?it/s]

hazard-category/val: 565 done, 0 missing | {'exact': 565}


product-category/val:   0%|          | 0/565 [00:00<?, ?it/s]

product-category/val: 565 done, 0 missing | {'exact': 565}


hazard/val:   0%|          | 0/60 [00:00<?, ?it/s]

...

product-category/test: 997 done, 0 missing | {'exact': 997}


hazard/test:   0%|          | 0/105 [00:00<?, ?it/s]

hazard/test: 997 done, 0 missing | {'single_candidate': 892, 'exact': 104, 'fallback': 1}


product/test:   0%|          | 0/415 [00:00<?, ?it/s]

product/test: 997 done, 0 missing | {'single_candidate': 582, 'exact': 408, 'fallback': 5, 'contained': 2}


In [47]:
def official_score(hazards_true, products_true, hazards_pred, products_pred):
    """SemEval-2025 Task 9 score: hazard macro-F1, plus product macro-F1 on samples
    with a correct hazard, averaged."""
    hazards_true, products_true = np.asarray(hazards_true), np.asarray(products_true)
    hazards_pred, products_pred = np.asarray(hazards_pred), np.asarray(products_pred)
    f1_hazards = f1_score(hazards_true, hazards_pred, average="macro")
    correct = hazards_pred == hazards_true
    f1_products = f1_score(products_true[correct], products_pred[correct], average="macro")
    return (f1_hazards + f1_products) / 2

def bert_top1(task, split):
    labels = json.loads((ENC_DIR / task / "labels.json").read_text())
    probs = np.load(ENC_DIR / task / f"probs_{split}.npy").astype(np.float32)
    return [labels[i] for i in probs.argmax(1)]

def llm_preds(task, split):
    results = load_jsonl(LLM_DIR / f"{task}_{split}.jsonl")
    ids = SPLIT_DF[split]["sample_id"].astype(int)
    missing = [i for i in ids if i not in results]
    assert not missing, f"{task}/{split}: {len(missing)} samples without LLM output; rerun Cell 14"
    return [results[i]["prediction"] for i in ids]

rows = []
for split in ["val", "test"]:
    df_ = SPLIT_DF[split]
    for system, get in [("BERT top-1", bert_top1), ("BERT + LLM", llm_preds)]:
        p = {t: get(t, split) for t in LABEL_COLS}
        rows.append({
            "split": split, "system": system,
            "ST1": official_score(df_["hazard-category"], df_["product-category"],
                                  p["hazard-category"], p["product-category"]),
            "ST2": official_score(df_["hazard"], df_["product"], p["hazard"], p["product"]),
        })
scores = pd.DataFrame(rows).round(4)
scores.to_csv(OUT_DIR / "final_scores.csv", index=False)
scores

,split,system,ST1,ST2
0,val,BERT top-1,0.8326,0.4658
1,val,BERT + LLM,0.6199,0.4776
2,test,BERT top-1,0.7348,0.4680
3,test,BERT + LLM,0.6238,0.4958
